# Telecom Auto-Documentation System

This notebook demonstrates how to use the analysis pipeline to scan a Python codebase and produce documentation artifacts.

## Setup

Install dependencies (run once):

In [ ]:
# Install required packages (uncomment if needed)
# %pip install -r requirements.txt

## 1 — Run the full pipeline

Point `target` at the root of any Python project you want to analyse.

In [ ]:
from main import run_pipeline

# Change this to the path of the codebase you want to analyse
TARGET_PATH = "."

artifacts = run_pipeline(
    target=TARGET_PATH,
    phases="1,2",          # Run Phase 1 (discovery) and Phase 2 (static analysis)
    log_level="INFO",
)

## 2 — Explore the artifacts

The returned `PhaseArtifacts` object holds all analysis results as Pydantic models.

### 2.1 File Inventory (Phase 1)

In [ ]:
inventory = artifacts.file_inventory
print(f"Project root : {inventory.project_root}")
print(f"Total files  : {inventory.total_files}")
print(f"Total lines  : {sum(f.line_count for f in inventory.files)}")
print()
print("First 10 files:")
for fm in inventory.files[:10]:
    print(f"  {fm.path}  ({fm.line_count} lines, {fm.encoding})")

### 2.2 AST Nodes (Phase 2)

In [ ]:
ast_nodes = artifacts.ast_nodes
print(f"Total AST nodes: {len(ast_nodes.nodes)}")
print()

# Show classes, functions, and methods
from collections import Counter
type_counts = Counter(n.node_type.value for n in ast_nodes.nodes)
for node_type, count in type_counts.most_common():
    print(f"  {node_type:10s}: {count}")

print("\nSample nodes:")
for node in ast_nodes.nodes[:5]:
    print(f"  [{node.node_type.value}] {node.name} (lines {node.line_start}-{node.line_end})")

### 2.3 Call Graph (Phase 2)

In [ ]:
call_graph = artifacts.call_graph
print(f"Call graph entries: {len(call_graph.entries)}")
print()
for entry in call_graph.entries[:5]:
    callees = ", ".join(c.function for c in entry.callees[:5])
    print(f"  {entry.caller.function} → {callees}")

### 2.4 Dependency Tree (Phase 2)

In [ ]:
dep_tree = artifacts.dependency_tree
print(f"Internal imports : {len(dep_tree.internal_imports)}")
print(f"External packages: {len(dep_tree.external_packages)}")
print()
if dep_tree.external_packages:
    print("External packages:")
    for pkg in dep_tree.external_packages:
        ver = pkg.version or "unknown"
        print(f"  {pkg.name} ({ver}) — used by {len(pkg.used_by_files)} file(s)")

### 2.5 Data Flow (Phase 2)

In [ ]:
data_flow = artifacts.data_flow
print(f"Data flow entries: {len(data_flow.entries)}")
print()
flow_counts = Counter(e.flow_type.value for e in data_flow.entries)
for ft, count in flow_counts.most_common():
    print(f"  {ft:15s}: {count}")

### 2.6 Component Map (Phase 2)

In [ ]:
comp_map = artifacts.component_map
print(f"Components: {len(comp_map.components)}")
print()
for comp in comp_map.components:
    ep = f"  entry_points={comp.entry_points}" if comp.entry_points else ""
    print(f"  {comp.component_name} — {comp.purpose_hint} ({len(comp.files)} files){ep}")

## 3 — Run individual phases

You can also run phases selectively or one at a time.

In [ ]:
# Run only Phase 1
inv_only = run_pipeline(target=TARGET_PATH, phases="1")
print(f"Discovered {inv_only.file_inventory.total_files} files")

## 4 — Export artifact as JSON

Every artifact is a Pydantic model, so you can serialise it to JSON easily.

In [ ]:
import json

# Pretty-print the file inventory
print(json.loads(artifacts.file_inventory.model_dump_json(indent=2))[:200] if False else "")

# Or save any artifact to a custom path
from pathlib import Path
out = Path("output/artifacts/file_inventory.json")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(artifacts.file_inventory.model_dump_json(indent=2))
print(f"Saved to {out}")

## 5 — Resume a previous run

If artifacts already exist on disk, pass `resume=True` to skip completed phases.

In [ ]:
artifacts = run_pipeline(
    target=TARGET_PATH,
    phases="1,2",
    resume=True,   # skips phases whose JSON artifacts already exist
)